# Document Pre-processing for Knowledge Tuning

## Overview

This notebook demonstrates a complete document preprocessing pipeline designed specifically for **knowledge tuning** with sdg-hub. 

## What This Notebook Does

This preprocessing pipeline transforms raw documents (PDFs, Word docs, etc.) into seed data for data generation:

1. **Document Parsing**: Converts raw documents to structured markdown format
2. **Chunking**: Splits documents into manageable chunks while preserving structure and context
3. **Seed Data Creation**: Formats chunks with in-context learning (ICL) templates for effective knowledge tuning

## Prerequisites

- We will use the existing InstructLab document parser (`docparser_v2.py`) and Document parsing configuration (`docling_v2_config.yaml`)
- Raw pdf documents in the `document_collection/` directory


In [ ]:
# Step 1: Document Processing Pipeline
# Define the directory containing raw documents to be processed
data_dir = "document_collection/"

# Run the document parser to convert documents to markdown
# - input-dir: Directory containing source documents
# - output-dir: Directory where processed markdown files will be saved
# - c: Configuration file specifying parsing parameters
!python ../docparser_v2.py --input-dir {data_dir} --output-dir {data_dir} -c ../docling_v2_config.yaml

In [ ]:
# Step 2: Install Required Dependencies
# Install packages needed for document processing and text chunking

%pip install docling markdown-it-py
%pip install --upgrade transformers

In [ ]:
# Step 3: Load Processed Documents
import glob
from pathlib import Path

# The parser step above produces markdown files in data_dir
list_md_files = sorted(glob.glob(f"{data_dir}/*.md"))

if not list_md_files:
    raise FileNotFoundError(f"No markdown files found in {data_dir}. Run Step 1 first.")

print(f"Found {len(list_md_files)} markdown files")
for fp in list_md_files:
    print(f"  - {Path(fp).name}")


In [ ]:
# Step 4: Text Chunking and Automatic ICL Generation

from markdown_it import MarkdownIt
from typing import List
from pathlib import Path
from dotenv import load_dotenv
from pydantic import SecretStr
from sdg_hub.core.blocks.llm import LLMChatBlock, PromptBuilderBlock
import datasets
import pandas as pd
import json
import os
import re


load_dotenv()


def chunk_markdown(text: str, max_tokens: int = 200, overlap: int = 50) -> List[str]:
    """Split markdown text into word chunks while preserving some overlap."""
    md = MarkdownIt()
    tokens = md.parse(text)

    blocks = []
    buf = []
    for tok in tokens:
        if tok.block and tok.type.endswith("_open"):
            buf = []
        elif tok.block and tok.type.endswith("_close"):
            if buf:
                blocks.append("\n".join(buf).strip())
                buf = []
        elif tok.content:
            buf.append(tok.content)
    if buf:
        blocks.append("\n".join(buf).strip())

    chunks = []
    current_words = []
    for block in blocks:
        words = block.split()
        for w in words:
            current_words.append(w)
            if len(current_words) >= max_tokens:
                chunks.append(" ".join(current_words))
                current_words = current_words[-overlap:] if overlap > 0 else []

    if current_words:
        chunks.append(" ".join(current_words))

    return chunks


def set_model_config_for_block(block):
    """Configure LLM block from .env using same provider logic as knowledge_generation."""
    model_provider = os.getenv("MODEL_PROVIDER", "hosted_vllm")

    if model_provider == "hosted_vllm":
        block.model = os.getenv("VLLM_MODEL", "hosted_vllm/meta-llama/Llama-3.3-70B-Instruct")
        block.api_base = os.getenv("VLLM_API_BASE", "http://localhost:8000/v1")
        block.api_key = SecretStr(os.getenv("VLLM_API_KEY", "EMPTY"))
        enable_reasoning = os.getenv("ENABLE_REASONING", "false").lower() in ("1", "true", "yes")
        block.enable_reasoning = enable_reasoning
    elif model_provider == "openai":
        block.model = os.getenv("OPENAI_MODEL", "openai/gpt-4")
        openai_key = os.getenv("OPENAI_API_KEY", "")
        block.api_key = SecretStr(openai_key) if openai_key else None
    elif model_provider == "openrouter":
        block.model = os.getenv("OPENAI_MODEL", "openai/gpt-4")
        openai_key = os.getenv("OPENAI_API_KEY", "")
        block.api_key = SecretStr(openai_key) if openai_key else None
        block.api_base = "https://openrouter.ai/api/v1"
    elif model_provider == "ollama":
        block.model = os.getenv("OLLAMA_MODEL", "ollama/gemma2")
        block.api_base = os.getenv("OLLAMA_API_BASE", "http://localhost:11434")
    elif model_provider == "maas":
        block.model = os.getenv("MAAS_MODEL")
        maas_key = os.getenv("MAAS_API_KEY", "")
        block.api_key = SecretStr(maas_key) if maas_key else None
        block.api_base = os.getenv("MAAS_API_BASE")
    else:
        raise ValueError(f"Unsupported MODEL_PROVIDER: {model_provider}")

    block.async_mode = True
    return block


def resolve_question_prompt_path() -> str:
    rel = Path("src/sdg_hub/flows/knowledge_infusion/enhanced_multi_summary_qa/extractive_summary/prompts/generate_question_list.yaml")
    candidates = [
        (Path.cwd() / rel).resolve(),
        (Path.cwd() / "../../../" / rel).resolve(),
        (Path.cwd() / "../../../../" / rel).resolve(),
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    raise FileNotFoundError("Could not resolve generate_question_list.yaml path")


def response_to_text(raw_output) -> str:
    """Return content, fallback to reasoning_content."""
    candidates = []
    if isinstance(raw_output, list):
        candidates = [x for x in raw_output if isinstance(x, dict)]
    elif isinstance(raw_output, dict):
        candidates = [raw_output]

    for obj in candidates:
        content = str(obj.get("content") or "").strip()
        if content:
            return content
        reasoning = str(obj.get("reasoning_content") or "").strip()
        if reasoning:
            return reasoning
    return ""


def truncate_words(text: str, max_words: int) -> str:
    words = text.split()
    if len(words) <= max_words:
        return text
    return " ".join(words[:max_words])


def compact_document_for_prompt(text: str, max_words: int = 4500) -> str:
    words = text.split()
    if len(words) <= max_words:
        return text

    head_size = int(max_words * 0.75)
    tail_size = max_words - head_size
    head = " ".join(words[:head_size])
    tail = " ".join(words[-tail_size:])
    return f"{head}\n\n[... CONTENT TRUNCATED ...]\n\n{tail}"


def parse_json_payload(text: str):
    if not text:
        return None
    try:
        payload = json.loads(text)
        if isinstance(payload, dict):
            return payload
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        payload = json.loads(match.group(0))
        if isinstance(payload, dict):
            return payload
    except json.JSONDecodeError:
        return None
    return None


def parse_questions(text: str):
    """Extract [QUESTION]... [END] pairs with regex fallback."""
    if not text:
        return []

    tag_questions = [
        q.strip()
        for q in re.findall(r"\[QUESTION\](.*?)\[END\]", text, flags=re.DOTALL | re.IGNORECASE)
        if q and q.strip()
    ]
    if tag_questions:
        return tag_questions

    # Fallback: use likely question lines
    line_candidates = []
    for line in text.splitlines():
        cleaned = line.strip(" -•\t")
        if cleaned.count("?") >= 1 and len(cleaned.split()) >= 5:
            line_candidates.append(cleaned)
    return line_candidates


def normalize_query(q: str) -> str:
    cleaned = str(q or "").replace("<Insert question here>", "").strip()
    cleaned = re.sub(r"\s+", " ", cleaned)
    return cleaned


def fallback_queries(document_outline: str):
    return [
        f"What are the main ideas presented in {document_outline}?",
        f"How can the key points in {document_outline} be applied in practice?",
        f"What limitations or trade-offs are discussed in {document_outline}?",
    ]


def generate_icl_base_for_document(doc_text: str, doc_name: str, base_block, max_retries: int = 2):
    doc_excerpt = compact_document_for_prompt(doc_text, max_words=4500)

    messages = [
        {"role": "system", "content": "You generate high-quality seed metadata for knowledge tuning."},
        {
            "role": "user",
            "content": (
                "Read the document and return ONLY valid JSON with the keys: "
                "document_outline, domain, icl_document, seed_query_1, seed_query_2, seed_query_3. "
                "Requirements: document_outline should be concise and representative; "
                "domain should be short (examples: finance, law, healthcare, technology, education, articles/essays); "
                "icl_document should be a representative excerpt useful for educational question generation; "
                "seed_query_1..3 should be self-contained educational questions about icl_document. "
                "Do not add markdown fences or extra text.\n\n"
                f"Document name: {doc_name}\n\n"
                f"[DOCUMENT]\n{doc_excerpt}"
            ),
        },
    ]

    for attempt in range(max_retries + 1):
        df = pd.DataFrame([{"icl_base_prompt": messages}])
        out = base_block.generate(df, _flow_max_concurrency=1)
        text = response_to_text(out.iloc[0]["raw_icl_base"])
        payload = parse_json_payload(text)
        if isinstance(payload, dict):
            document_outline = str(payload.get("document_outline") or "").strip()
            domain = str(payload.get("domain") or "").strip()
            icl_doc = str(payload.get("icl_document") or "").strip()
            seed_q1 = normalize_query(payload.get("seed_query_1", ""))
            seed_q2 = normalize_query(payload.get("seed_query_2", ""))
            seed_q3 = normalize_query(payload.get("seed_query_3", ""))

            if document_outline and domain and icl_doc:
                icl_doc = truncate_words(icl_doc, 1600)
                seeds = [seed_q1, seed_q2, seed_q3]
                seed_fallback = fallback_queries(document_outline)
                for i in range(3):
                    if not seeds[i]:
                        seeds[i] = seed_fallback[i]
                return {
                    "document_outline": document_outline,
                    "domain": domain,
                    "icl_document": icl_doc,
                    "seed_query_1": seeds[0],
                    "seed_query_2": seeds[1],
                    "seed_query_3": seeds[2],
                }

        print(f"  - Retry icl base for {doc_name}: attempt {attempt + 1}/{max_retries + 1}")

    # Final fallback for this document
    outline = Path(doc_name).stem.replace("_", " ").strip() or "Document"
    seeds = fallback_queries(outline)
    return {
        "document_outline": outline,
        "domain": "articles/essays",
        "icl_document": truncate_words(doc_text, 1600),
        "seed_query_1": seeds[0],
        "seed_query_2": seeds[1],
        "seed_query_3": seeds[2],
    }


def generate_final_queries(base_icl: dict, prompt_path: str, prompt_block, question_block, max_retries: int = 2):
    seed_qs = [base_icl["seed_query_1"], base_icl["seed_query_2"], base_icl["seed_query_3"]]

    row = {
        "domain": base_icl["domain"],
        "document": base_icl["icl_document"],
        "document_outline": base_icl["document_outline"],
        "icl_document": base_icl["icl_document"],
        "icl_query_1": seed_qs[0],
        "icl_query_2": seed_qs[1],
        "icl_query_3": seed_qs[2],
    }

    prompt_df = prompt_block.generate(pd.DataFrame([row]))

    parsed = []
    for attempt in range(max_retries + 1):
        out = question_block.generate(prompt_df.copy(), _flow_max_concurrency=1)
        text = response_to_text(out.iloc[0]["raw_question_list"])
        parsed = [normalize_query(q) for q in parse_questions(text)]
        parsed = [q for q in parsed if q]
        if parsed:
            break
        print(f"  - Retry question generation for {base_icl['document_outline']}: attempt {attempt + 1}/{max_retries + 1}")

    # Deduplicate while preserving order
    deduped = []
    for q in parsed + seed_qs:
        if q and q not in deduped:
            deduped.append(q)

    while len(deduped) < 3:
        deduped.append(fallback_queries(base_icl["document_outline"])[len(deduped)])

    return deduped[:3]


# Resolve official prompt path
question_prompt_path = resolve_question_prompt_path()
print(f"Using prompt: {question_prompt_path}")

# Build reusable LLM blocks
icl_base_block = LLMChatBlock(
    block_name="generate_icl_base",
    input_cols=["icl_base_prompt"],
    output_cols=["raw_icl_base"],
    temperature=0.3,
    max_tokens=1800,
)
icl_base_block = set_model_config_for_block(icl_base_block)

question_prompt_block = PromptBuilderBlock(
    block_name="build_question_prompt",
    input_cols=[
        "domain",
        "document",
        "document_outline",
        "icl_document",
        "icl_query_1",
        "icl_query_2",
        "icl_query_3",
    ],
    output_cols=["question_generation_prompt"],
    prompt_config_path=question_prompt_path,
    format_as_messages=True,
)

question_generation_block = LLMChatBlock(
    block_name="generate_icl_questions",
    input_cols=["question_generation_prompt"],
    output_cols=["raw_question_list"],
    temperature=0.7,
    max_tokens=512,
)
question_generation_block = set_model_config_for_block(question_generation_block)

# Process every markdown document and replicate per-doc ICL across its chunks
seed_rows = []
processed_docs = 0
skipped_docs = 0

for md_fp in list_md_files:
    doc_name = Path(md_fp).name
    print(f"\nProcessing: {doc_name}")
    try:
        with open(md_fp, "r", encoding="utf-8") as f:
            doc_text = f.read()

        chunks = chunk_markdown(doc_text, max_tokens=5000, overlap=1000)
        if not chunks:
            print("  - Skipping: no chunks generated")
            skipped_docs += 1
            continue

        base_icl = generate_icl_base_for_document(doc_text, doc_name, icl_base_block, max_retries=2)
        final_queries = generate_final_queries(
            base_icl,
            question_prompt_path,
            question_prompt_block,
            question_generation_block,
            max_retries=2,
        )

        for chunk in chunks:
            seed_rows.append(
                {
                    "document": chunk,
                    "document_outline": base_icl["document_outline"],
                    "icl_document": base_icl["icl_document"],
                    "icl_query_1": final_queries[0],
                    "icl_query_2": final_queries[1],
                    "icl_query_3": final_queries[2],
                    "domain": base_icl["domain"],
                }
            )

        print(f"  - Chunks: {len(chunks)}")
        print(f"  - Outline: {base_icl['document_outline']}")
        print(f"  - Domain: {base_icl['domain']}")
        processed_docs += 1
    except Exception as exc:
        print(f"  - Skipping due to error: {exc}")
        skipped_docs += 1

if not seed_rows:
    raise ValueError("No seed rows generated. Check data_dir and model configuration.")

# Build and validate final dataset schema
required_cols = [
    "document",
    "document_outline",
    "icl_document",
    "icl_query_1",
    "icl_query_2",
    "icl_query_3",
    "domain",
]

seed_data = datasets.Dataset.from_list(seed_rows).select_columns(required_cols)

for col in ["document_outline", "icl_document", "icl_query_1", "icl_query_2", "icl_query_3", "domain"]:
    empty_count = sum(1 for v in seed_data[col] if not str(v).strip())
    if empty_count > 0:
        raise ValueError(f"Column {col} has {empty_count} empty values after ICL generation")

# Save the seed data to a JSONL file for downstream use
seed_data.to_json("seed_data.jsonl", orient="records", lines=True)

print("\n" + "=" * 60)
print("ICL AUTOMATIC GENERATION SUMMARY")
print("=" * 60)
print(f"Processed documents: {processed_docs}")
print(f"Skipped documents:   {skipped_docs}")
print(f"Total output rows:   {len(seed_data)}")
print("Saved: seed_data.jsonl")


### Next Steps:
- The seed_data.jsonl file is now ready for the knowledge tuning pipeline.
- You can now refer to the [knowledge generation](knowledge_generation.ipynb) notebook